# Get Data

In [1]:
pip install pytrends

## Google Trends Data

In [2]:
import pandas as pd

# ----------------------------
# Google Trends Time
# ----------------------------

trends_time = pd.read_csv("src_google_trends_time_raw.csv")

trends_time["date"] = pd.to_datetime(trends_time["date"])
trends_time["year"] = trends_time["date"].dt.year
trends_time["month_num"] = trends_time["date"].dt.month
trends_time["year_month"] = trends_time["date"].dt.strftime("%Y-%m")

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall"
}

trends_time["season"] = trends_time["month_num"].map(season_map)

trends_time = trends_time.rename(columns={
    "ice cream": "ice_cream"
})

trends_time.to_csv("src_google_trends_time_clean.csv", index=False)


# ----------------------------
# Google Trends State
# ----------------------------

trends_state = pd.read_csv("src_google_trends_state_raw.csv")

# normalize column names
trends_state.columns = (
    trends_state.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# rename first column to state if needed
if "state" not in trends_state.columns:
    trends_state = trends_state.rename(columns={
        trends_state.columns[0]: "state"
    })

trends_state = trends_state.rename(columns={
    "ice_cream": "ice_cream",
    "ice_cream_": "ice_cream",
    "ice": "ice_cream"
})

# make sure product columns are valid
trends_state = trends_state.rename(columns={
    "ice_cream": "ice_cream",
    "pizza": "pizza"
})

trends_state["state"] = trends_state["state"].str.strip()

trends_state.to_csv("src_google_trends_state_clean.csv", index=False)

print("Exported:")
print("src_google_trends_time_clean.csv")
print("src_google_trends_state_clean.csv")

Exported:
src_google_trends_time_clean.csv
src_google_trends_state_clean.csv


In [3]:
print(trends_time.shape)
print(trends_time.head())

(72, 8)
        date  ice_cream  pizza  isPartial  year  month_num year_month  season
0 2020-01-01         11     87      False  2020          1    2020-01  Winter
1 2020-02-01         12     90      False  2020          2    2020-02  Winter
2 2020-03-01         12     86      False  2020          3    2020-03  Spring
3 2020-04-01         14     97      False  2020          4    2020-04  Spring
4 2020-05-01         19     95      False  2020          5    2020-05  Spring


In [4]:
print(trends_state.shape)
print(trends_state.head())

(51, 3)
        state  ice_cream  pizza
0     Alabama         18     82
1      Alaska         14     86
2     Arizona         15     85
3    Arkansas         14     86
4  California         20     80


In [10]:
df = pd.read_csv("src_google_trends_state_clean.csv")

print(df.columns)
print(df.head())

# remove unnamed/blank columns
df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed|None", case=False, na=False)]

# clean column names
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# rename if needed
if "ice_cream" not in df.columns:
    df = df.rename(columns={"ice cream": "ice_cream"})

df.to_csv("src_google_trends_state_mysql.csv", index=False)

print(df.columns)
print("Exported src_google_trends_state_mysql.csv")

Index(['state', 'ice_cream', 'pizza'], dtype='object')
        state  ice_cream  pizza
0     Alabama         18     82
1      Alaska         14     86
2     Arizona         15     85
3    Arkansas         14     86
4  California         20     80
Index(['state', 'ice_cream', 'pizza'], dtype='object')
Exported src_google_trends_state_mysql.csv


## NOAA Climate Data

In [5]:
import pandas as pd

# ----------------------------
# 1. Load NOAA statewide temp txt file
# ----------------------------

# Dataset from https://www.ncei.noaa.gov/pub/data/cirs/climdiv/
file_path = "climdiv-tmpcst-v1.0.0-20260506.txt"

noaa = pd.read_fwf(
    file_path,
    widths=[2, 2, 2, 4] + [7] * 12,
    header=None,
    dtype=str
)

noaa.columns = [
    "state_code",
    "division_code",
    "element_code",
    "year",
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec"
]

# ----------------------------
# 2. Keep statewide records only
# ----------------------------

noaa = noaa[noaa["division_code"] == "00"].copy()

# ----------------------------
# 3. Convert NOAA state codes → state names
# ----------------------------

state_map = {
    "01": "Alabama",
    "02": "Arizona",
    "03": "Arkansas",
    "04": "California",
    "05": "Colorado",
    "06": "Connecticut",
    "07": "Delaware",
    "08": "Florida",
    "09": "Georgia",
    "10": "Idaho",
    "11": "Illinois",
    "12": "Indiana",
    "13": "Iowa",
    "14": "Kansas",
    "15": "Kentucky",
    "16": "Louisiana",
    "17": "Maine",
    "18": "Maryland",
    "19": "Massachusetts",
    "20": "Michigan",
    "21": "Minnesota",
    "22": "Mississippi",
    "23": "Missouri",
    "24": "Montana",
    "25": "Nebraska",
    "26": "Nevada",
    "27": "New Hampshire",
    "28": "New Jersey",
    "29": "New Mexico",
    "30": "New York",
    "31": "North Carolina",
    "32": "North Dakota",
    "33": "Ohio",
    "34": "Oklahoma",
    "35": "Oregon",
    "36": "Pennsylvania",
    "37": "Rhode Island",
    "38": "South Carolina",
    "39": "South Dakota",
    "40": "Tennessee",
    "41": "Texas",
    "42": "Utah",
    "43": "Vermont",
    "44": "Virginia",
    "45": "Washington",
    "46": "West Virginia",
    "47": "Wisconsin",
    "48": "Wyoming",
    "49": "Alaska",
    "50": "Hawaii"
}

noaa["state"] = noaa["state_code"].map(state_map)

# remove non-state aggregate rows
noaa = noaa[noaa["state"].notna()].copy()

# ----------------------------
# 4. Convert temperature columns to numeric
# ----------------------------

month_cols = [
    "jan", "feb", "mar", "apr", "may", "jun",
    "jul", "aug", "sep", "oct", "nov", "dec"
]

for col in month_cols:
    noaa[col] = pd.to_numeric(noaa[col], errors="coerce")

noaa["year"] = pd.to_numeric(noaa["year"], errors="coerce")

# ----------------------------
# 5. Convert wide → long format
# ----------------------------

noaa_long = noaa.melt(
    id_vars=["state", "year"],
    value_vars=month_cols,
    var_name="month",
    value_name="avg_temp"
)

# ----------------------------
# 6. Add month numbers
# ----------------------------

month_map = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12
}

noaa_long["month_num"] = noaa_long["month"].map(month_map)

# ----------------------------
# 7. Create date + join key
# ----------------------------

noaa_long["date"] = pd.to_datetime({
    "year": noaa_long["year"],
    "month": noaa_long["month_num"],
    "day": 1
})

noaa_long["year_month"] = (
    noaa_long["date"]
    .dt.strftime("%Y-%m")
)

# ----------------------------
# 8. Keep recent years only
# ----------------------------

noaa_long = noaa_long[noaa_long["year"] >= 2020]

# ----------------------------
# 9. Export clean CSV
# ----------------------------

noaa_long.to_csv(
    "src_noaa_temperature_clean.csv",
    index=False
)

print(noaa_long.shape)
print(noaa_long.head())
print(noaa_long.isna().sum())

(1008, 7)
       state  year month  avg_temp  month_num       date year_month
125  Alabama  2020   jan      27.3          1 2020-01-01    2020-01
126  Alabama  2021   jan      26.6          1 2021-01-01    2021-01
127  Alabama  2022   jan      23.9          1 2022-01-01    2022-01
128  Alabama  2023   jan      23.2          1 2023-01-01    2023-01
129  Alabama  2024   jan      25.4          1 2024-01-01    2024-01
state         0
year          0
month         0
avg_temp      0
month_num     0
date          0
year_month    0
dtype: int64


## Yelp

In [6]:
import pandas as pd
import json

# ----------------------------------------
# 1. Load Yelp business dataset
# ----------------------------------------

# Dataset from https://business.yelp.com/data/resources/open-dataset/

rows = []

with open("yelp_academic_dataset_business.json", "r") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

print(df.shape)

# ----------------------------------------
# 2. Convert state abbreviations
# ----------------------------------------

state_map = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
    "DC": "District of Columbia",
    "PR": "Puerto Rico"
}

df["state"] = df["state"].map(state_map)

# ----------------------------------------
# 3. Filter Pizza + Ice Cream businesses
# ----------------------------------------

food_df = df[
    df["categories"].str.contains(
        "Pizza|Ice Cream",
        case=False,
        na=False
    )
].copy()

# ----------------------------------------
# 4. Keep useful columns only
# ----------------------------------------

food_df = food_df[[
    "business_id",
    "name",
    "city",
    "state",
    "stars",
    "review_count",
    "categories"
]]

print(food_df.shape)
print(food_df.head())

# ----------------------------------------
# 5. Export clean Yelp dataset
# ----------------------------------------

food_df.to_csv(
    "src_yelp_clean.csv",
    index=False
)

print("Yelp clean dataset exported.")

(150346, 14)
(9714, 7)
               business_id              name          city         state  \
5   CF33F8-E6oudUQ46HnavjQ    Sonic Drive-In  Ashland City     Tennessee   
9   bBDDEgkFA1Otx9Lfe7BZUQ    Sonic Drive-In     Nashville     Tennessee   
29  sqSqqLy0sN8n2IZrAbzidQ    Domino's Pizza   White House     Tennessee   
31  Mjboz24M9NlBeiOJKLEd_Q  DeSandro on Main  Philadelphia  Pennsylvania   
33  kV_Q1oqis8Qli8dUoGpTyQ     Ardmore Pizza       Ardmore  Pennsylvania   

    stars  review_count                                         categories  
5     2.0             6  Burgers, Fast Food, Sandwiches, Food, Ice Crea...  
9     1.5            10  Ice Cream & Frozen Yogurt, Fast Food, Burgers,...  
29    3.5             8      Pizza, Chicken Wings, Sandwiches, Restaurants  
31    3.0            41                    Pizza, Restaurants, Salad, Soup  
33    3.5           109                                 Pizza, Restaurants  
Yelp clean dataset exported.


## U.S. Census Population Data

In [7]:
import pandas as pd

# Dataset from https://www.census.gov/data/tables/time-series/demo/popest/2020s-state-total.html

cols = [
    "SUMLEV","REGION","DIVISION","STATE","NAME",
    "ESTIMATESBASE2020",
    "POPESTIMATE2020","POPESTIMATE2021","POPESTIMATE2022",
    "POPESTIMATE2023","POPESTIMATE2024","POPESTIMATE2025",
    "NPOPCHG_2020","NPOPCHG_2021","NPOPCHG_2022",
    "NPOPCHG_2023","NPOPCHG_2024","NPOPCHG_2025",
    "PPOPCHG_2020","PPOPCHG_2021","PPOPCHG_2022",
    "PPOPCHG_2023","PPOPCHG_2024","PPOPCHG_2025"
]

df = pd.read_csv(
    "NST-EST2025-POPCHG2020-2025.csv",
    header=None,
    skiprows=1
)

df = df.iloc[:, :len(cols)]
df.columns = cols

df["SUMLEV"] = df["SUMLEV"].astype(str).str.zfill(3)

region_map = {
    1: "Northeast",
    2: "Midwest",
    3: "South",
    4: "West"
}

df["REGION"] = pd.to_numeric(df["REGION"], errors="coerce")

df_states = df[df["SUMLEV"] == "040"][[
    "STATE",
    "NAME",
    "REGION",
    "POPESTIMATE2025"
]]

division_map = {
    1: "New England",
    2: "Middle Atlantic",
    3: "East North Central",
    4: "West North Central",
    5: "South Atlantic",
    6: "East South Central",
    7: "West South Central",
    8: "Mountain",
    9: "Pacific"
}

df["DIVISION"] = pd.to_numeric(df["DIVISION"], errors="coerce")

df_states = df[df["SUMLEV"] == "040"][[
    "STATE",
    "NAME",
    "REGION",
    "DIVISION",
    "POPESTIMATE2025"
]]

df_states["region"] = df_states["REGION"].map(region_map)
df_states["division"] = df_states["DIVISION"].map(division_map)

df_states = df_states.rename(columns={
    "STATE": "state_fips",
    "NAME": "state",
    "POPESTIMATE2025": "population_2025"
})

df_states = df_states.drop(columns=["REGION", "DIVISION"])

print(df_states.shape)
print(df_states.head())

(52, 5)
    state_fips       state  population_2025 region            division
14           1     Alabama          5193088  South  East South Central
15           2      Alaska           737270   West             Pacific
16           4     Arizona          7623818   West            Mountain
17           5    Arkansas          3114791  South  West South Central
18           6  California         39355309   West             Pacific


In [8]:
df_states.to_csv(
    "src_census_population_regions_clean.csv",
    index=False
)

print("CSV exported successfully.")

CSV exported successfully.


In [9]:
import pandas as pd
import unicodedata
import csv

food_df = pd.read_csv("src_yelp_clean.csv", encoding="utf-8")

def clean_ascii(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = unicodedata.normalize("NFKD", x)
    x = x.encode("ascii", errors="ignore").decode("ascii")
    return x.replace("\n", " ").replace("\r", " ").replace("\t", " ")

for col in food_df.columns:
    if food_df[col].dtype == "object":
        food_df[col] = food_df[col].apply(clean_ascii)

food_df.to_csv(
    "src_yelp_clean_mysql_ascii.csv",
    index=False,
    encoding="ascii",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n"
)

print(food_df.shape)
print("Exported src_yelp_clean_mysql_ascii.csv")

(9714, 7)
Exported src_yelp_clean_mysql_ascii.csv
